[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NDesassis/notebooks/blob/main/sampling_with_diffusion.ipynb)

Author: Gabriel V. Cardoso (freely adapted)

https://gabrielvc.github.io/

This notebook has two parts, one for the non-conditional generation and one for the posterior sampling.

# I - Unconditional generation

The goal of this part is to implement the backward procedure of the diffusion generative model.

## Recall: Forward process

The forward process is the Markov Chain defined as

$$X_{t+1} = X_{t} + \sigma_t Z_t\;,$$

with $X_0 \sim p_{0}$ and $Z_t$ iid Standard gaussian distributions.


## Recall: Backward process

The backward process is the Markov chain that best approximates the joint law of the forward process, which is characterized by the two moments $\mathbb{E}[X_t | X_{t+1}=x]$ and $\mathbb{V}[X_t | X_{t+1}=x]$. We consider that we have a denoiser network $D_{\theta}$ that approximates $\mathbb{E}[X_t | X_{t+1}=x]$.

In [ ]:
import torch
import torch
import torch.nn.functional as F
from torchvision.transforms.functional import pil_to_tensor
from torchvision import transforms

from functools import partial
from typing import Callable
from tqdm import tqdm
import math
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

import math
from diffusers import UNet2DModel, DDPMScheduler
from urllib import response
import numpy as np

import cv2


device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
def ddpm(
    sigmas: torch.Tensor,
    initial_samples: torch.Tensor,
    denoiser_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor]
) -> torch.Tensor:
    # Implement your code here!
    samples = initial_samples
    for sigma_tm1, sigma_t in tqdm(zip(reversed(sigmas[:-1]), reversed(sigmas[1:]))):
        denoised_x0 = ...
        mean = ...
        std = ...
        samples = ...
    return samples



## First example: Mixture of Gaussians

We are first going to use a Mixture of Gaussians as the target distribution. This is particularly interesting from the visualization point of view and also because we have the "true score" function.


In [ ]:
def get_random_unitary(n=20, dim=2):
    seeds = torch.randn((n, dim, dim))
    U, _, _ = torch.linalg.svd(seeds)
    S = torch.stack([torch.diag(s) for s in torch.rand((n, dim))])
    return U @ S @ U.mT

means = torch.stack(
    torch.meshgrid(
        torch.linspace(-1, 1, 5),
        torch.linspace(-1, 1, 5),
    )
).flatten(1, 2).T
covars = get_random_unitary(n=means.shape[0], dim=means.shape[1])*(0.15**2)
weights = torch.rand(means.shape[0])
weights = weights / weights.sum()
def make_noised_mixture(means, covars, weights, sigma):
    return torch.distributions.MixtureSameFamily(
        component_distribution=torch.distributions.MultivariateNormal(
            loc=means, covariance_matrix=covars + torch.eye(covars.shape[-1])[None]*sigma**2
        ),
        mixture_distribution=torch.distributions.Categorical(probs=weights)
    )

def denoiser_fn_mixture(x, sigma):
    score = torch.func.grad(lambda y: make_noised_mixture(means, covars, weights, sigma).log_prob(y).sum())(x)
    return x + (sigma**2) * score

fig, ax = plt.subplots(1, 1)
fig.suptitle(r"$p_{0}$")
ax.scatter(*make_noised_mixture(means, covars, weights, sigma=0).sample((10_000,)).detach().cpu().T, alpha=.05)
ax.set_xlim([-1.3, 1.3])
ax.set_ylim([-1.3, 1.3])
fig.show()

In [ ]:
sigmas = torch.linspace(0.002**(1/3), 80**(1/3), 1000)**3
generated_samples = ddpm(
    initial_samples = torch.randn((10_000, 2))*((sigmas[-1]**2 + 1)**.5),
    sigmas = sigmas,
    denoiser_fn=denoiser_fn_mixture
)
fig, ax = plt.subplots(1, 1)
fig.suptitle(r"$p_{0}$")
ax.scatter(*make_noised_mixture(means, covars, weights, sigma=0).sample((10_000,)).T, alpha=.05, label="True")
ax.scatter(*generated_samples.T, alpha=.05, label="Generated")
ax.set_xlim([-1.3, 1.3])
ax.set_ylim([-1.3, 1.3])
fig.legend()
fig.show()

## Second example : the celeb A data set

For this data set, we download a pre-trained denoiser.

In [ ]:

#We download the denoiser trained on celebahq-256 data set
model_id = "google/ddpm-celebahq-256"
scheduler = DDPMScheduler.from_pretrained(model_id)
model = UNet2DModel.from_pretrained(model_id).to(device).eval().requires_grad_(False)


In [ ]:

# Test the denoiser on a real image
url = "https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=256&h=256&fit=crop"
image_raw = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

preprocess = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]) # Crucial pour le U-Net (plage -1 à 1)
])


def postprocess(tensor):
    tensor = (tensor.detach().cpu().clamp(-1, 1) + 1) / 2
    return tensor.permute(0, 2, 3, 1).squeeze().numpy()


def denoiser_fn_hf(x, sigma):
    # This part belows convert the variance exploding framework to the variance preserving.

    eq_alpha = 1 / (sigma**2 + 1)
    get_closest = torch.abs(scheduler.alphas_cumprod.to(eq_alpha.device) - eq_alpha).argmin().item()
    closest_alpha = scheduler.alphas_cumprod[get_closest]
    eq_x = x / (sigma**2 + 1)**.5

    out_model = model(eq_x, get_closest).sample
    pred_x0 = ((eq_x  -  ((1 - closest_alpha)**.5)*out_model) / (closest_alpha**.5)).clip(-1, 1)
    return pred_x0

In [ ]:
target_sigma = torch.tensor([0.95], device=device)
input_tensor = preprocess(image_raw).unsqueeze(0).to(device)
# 2. Créer l'image bruitée (selon la logique Forward du notebook)
# x_noisy = x_0 + sigma * epsilon
noise = torch.randn_like(input_tensor)
noisy_image = input_tensor + target_sigma * noise


denoised_result = denoiser_fn_hf(noisy_image, target_sigma)

# 4. Affichage des 3 étapes
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(postprocess(input_tensor))
ax[0].set_title("1. Originale (x0)")
ax[0].axis('off')

ax[1].imshow(postprocess(noisy_image))
ax[1].set_title(f"2. Bruitée (sigma={np.round(target_sigma.item(),2)})")
ax[1].axis('off')

ax[2].imshow(postprocess(denoised_result))
ax[2].set_title("3. Débruitée (via denoiser_fn_hf)")
ax[2].axis('off')

plt.show()

In [ ]:
sigmas = (1/scheduler.alphas_cumprod - 1)**.5
generated_samples = ddpm(
    initial_samples = torch.randn((1, 3, 256, 256), device=device)*((sigmas[-1]**2 + 1)**.5),
    sigmas = sigmas,
    denoiser_fn=denoiser_fn_hf
)

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.imshow(((generated_samples[0] + 1)/2).permute(1, 2, 0).cpu())
ax.set_axis_off()
fig.tight_layout()
fig.show()

# II. Posterior sampling

In [ ]:
# Here we choose the type of direct problem. blur = True for deblurring, blur = False for inpainting.
blur = True

### Observation process modeling

In [ ]:

class GaussianBlurPyTorch:
    def __init__(self, kernel_size=15, sigma=None, channels=3, device="cuda"):
        self.kernel_size = kernel_size
        self.channels = channels
        self.device = device
        
        if sigma is None or sigma <= 0:
            sigma = 0.3 * ((kernel_size - 1) * 0.5 - 1) + 0.8
        
        x = torch.linspace(-(kernel_size // 2), kernel_size // 2, kernel_size)
        gauss = torch.exp(-x**2 / (2 * sigma**2))
        gauss = gauss / gauss.sum() 
        
       
        kernel_2d = gauss[:, None] @ gauss[None, :]
        
     
        self.kernel = kernel_2d.expand(channels, 1, kernel_size, kernel_size).to(device)

    def __call__(self, img_tensor):
        """
        img_tensor: [B, 3, 256, 256] normalisé entre -1 et 1
        """
        padding = self.kernel_size // 2
        return F.conv2d(img_tensor, self.kernel, groups=self.channels, padding=padding)
    
blur_op = GaussianBlurPyTorch(kernel_size=51, sigma=10.0, device="cuda")

class InpaintOperator:
    def __init__(self, mask):
        self.mask = mask 

    def __call__(self, x):
     
        return self.mask * x

mask = torch.ones((1, 1, 256, 256)).to(device)
# black out the center (ex: 128x128 pixels)
mask[:, :, 64:192, 64:192] = 0.0

inpaint_op = InpaintOperator(mask)
if blur:
    operator = blur_op
else:
    operator = inpaint_op

Get the image

In [ ]:


def prepare_observation(blur, device="cuda"):
    # 1. Load image  (OpenCV)
    if blur:
        url = "https://github.com/NDesassis/notebooks/raw/main/blurred_image.jpg"
    else:
        url = "https://github.com/NDesassis/notebooks/raw/main/inpaint_image.jpg"
   
    response = requests.get(url)
    image_array = np.frombuffer(response.content, dtype=np.uint8)
    img_bgr = cv2.imdecode(image_array, cv2.IMREAD_COLOR)
   
    
    if img_bgr is None:
        raise FileNotFoundError(f"Not possible to read {url}")

    # 2. Conversion BGR -> RGB 
    # (U-Net has been trained on RGB images, the inversion would completely mess up the gradient)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. Reshaping for safety
    img_rgb = cv2.resize(img_rgb, (256, 256), interpolation=cv2.INTER_AREA)

    # 4. Normalisation [-1, 1] 
    # Formula (x / 127.5) - 1.0 make the corresponence 0->-1, 127->0 et 255->1.
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float()
    observation = (img_tensor / 127.5) - 1.0

    # 5. Add batch dimension and move to device
    observation = observation.unsqueeze(0).to(device)
    
    return observation


observation = prepare_observation(blur)


In [ ]:
img_to_show = postprocess(observation)
plt.imshow(img_to_show)
plt.axis("off")

In [ ]:

## Helper functions
def img_to_tensor(img):
    return (pil_to_tensor(img) / 255 - 0.5)*2

def tensors_to_img(dt, rescale=True):
    if rescale:
        images = ((dt / 2 + 0.5).clamp(0, 1)*255).round()
    else:
        images = dt
    images = images.cpu().permute(0, 2, 3, 1).numpy()
    images = [Image.fromarray(image.astype(np.uint8)) for image in images]
    return images

In [ ]:
def denoiser_dps(x, sigma, loglik, constant):
    def denoiser_and_lik(x):
        pred_x0 = denoiser_fn_hf(x[None], sigma)[0]
        loglik_val = loglik(pred_x0)
        return loglik_val, pred_x0

    grad_loglik, (loglik_val, pred_x0) = torch.func.vmap(torch.func.grad_and_value(denoiser_and_lik, has_aux=True))(x)
    error_val = (-loglik_val)**.5
    zeta = (constant / error_val)[:, None, None, None]
    return pred_x0 + sigma**2 * zeta * grad_loglik

In [ ]:


n_tot_samples = 5
batch_size = 1
sigma_err = .05
n_batches = n_tot_samples // batch_size + (n_tot_samples % batch_size !=0)
sample_size = model.config.sample_size
sigmas = ((1/scheduler.alphas_cumprod - 1)**.5)[::10]

dps_imgs = []
dps_reconstructions = []
for _ in range(n_batches):
    noise = torch.randn((batch_size, 3, sample_size, sample_size), device = device) * (1 + sigmas[-1]**2)**.5
    dps_out = ddpm(
        sigmas=sigmas,
        initial_samples=noise,
        denoiser_fn=partial(
            denoiser_dps,
            loglik=lambda x: ...,
            constant=.05
        )
        )
    dps_imgs += tensors_to_img(dps_out)
    dps_reconstructions += tensors_to_img(operator(dps_out))

In [ ]:
fig, axs = plt.subplots(1,1+ n_tot_samples, figsize=(12, 6))


axs[0].imshow(postprocess(observation))
axs[0].set_title("1. Observation")
axs[0].axis('off')


for i in range(0, n_tot_samples):
    axs[i+1].imshow(dps_imgs[i]) 
    axs[i+1].set_title(f"{i+1}")
    axs[i+1].axis('off')

fig.tight_layout() #
plt.show()